In [ ]:
import pandas as pd
import requests
import io

print("Consultando API de Datos Argentina (IPC Nacional)...")

# ID de la serie oficial IPC Nivel General (Serie base 2016)
ID_IPC_NACIONAL = "101.1_I2NG_2016_M_22"
url_ipc = f"https://apis.datos.gob.ar/series/api/series/?ids={ID_IPC_NACIONAL}&start_date=2024-01&format=csv"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    response = requests.get(url_ipc, headers=headers)
    if response.status_code == 200:
        df = pd.read_csv(io.StringIO(response.text))
        df.columns = ['Fecha', 'IPC_Nacional_Indice']

        # 1. Convertir fecha a datetime
        df['Fecha'] = pd.to_datetime(df['Fecha'])

        # 2. Calcular Inflacion Mensual (%)
        df['Inflacion_Mensual (%)'] = df['IPC_Nacional_Indice'].pct_change() * 100

        # 3. Calcular Variacion Interanual (%) (comparado con el mismo mes del año anterior)
        df['Inflacion_Interanual (%)'] = df['IPC_Nacional_Indice'].pct_change(periods=12) * 100

        # 4. Asignar año y mes para pivoteo o filtrado comparativo 2025 vs 2026
        df['Año'] = df['Fecha'].dt.year
        df['Mes'] = df['Fecha'].dt.strftime('%m - %B')

        # Filtrar solo el rango 2025-2026
        df_comparativo = df[df['Año'].isin([2025, 2026])].copy()

        print("\n--- Vista de los últimos datos registrados ---")
        display(df_comparativo.tail(10))

        # Exportar CSV preparado para Datawrapper / Flourish
        df_comparativo.to_csv("ipc_nacional_2025_2026.csv", index=False, encoding='utf-8-sig')
        print("\nArchivo 'ipc_nacional_2025_2026.csv' exportado correctamente.")

    else:
        print(f"Error al consultar la API. Código de estado: {response.status_code}")
except Exception as e:
    print(f"Ocurrió un error en la conexión: {e}")

Consultando API de Datos Argentina (IPC Nacional)...

--- Vista de los últimos datos registrados ---


,Fecha,IPC_Nacional_Indice,Inflacion_Mensual (%),Inflacion_Interanual (%),Año,Mes
22,2025-11-01,9840.8457,2.481975,31.828820,2025,11 - November
23,2025-12-01,10115.6707,2.792697,31.735483,2025,12 - December
24,2026-01-01,10395.7458,2.768725,32.697853,2026,01 - January
25,2026-02-01,10667.6896,2.615914,33.192496,2026,02 - February
26,2026-03-01,11031.5404,3.410774,32.623706,2026,03 - March
27,2026-04-01,11336.5576,2.764956,32.633001,2026,04 - April
28,2026-05-01,11594.5499,2.275755,33.645591,2026,05 - May
29,2026-06-01,11810.9464,1.866364,33.438605,2026,06 - June
30,2026-07-01,12078.7372,2.267310,33.961758,2026,07 - July
31,2026-08-01,12276.8120,1.639863,33.657071,2026,08 - August



Archivo 'ipc_nacional_2025_2026.csv' exportado correctamente.


In [ ]:
# Reestructurar datos para visualización en columnas paralelas
df_pivot = df_comparativo.pivot(index='Mes', columns='Año', values='Inflacion_Mensual (%)')

# Exportar para Datawrapper / Flourish
df_pivot.to_csv("ipc_comparativo_pivot.csv")
display(df_pivot)

Año,2025,2026
Mes,,
01 - January,2.023411,2.768725
02 - February,2.234825,2.615914
03 - March,3.854277,3.410774
04 - April,2.757754,2.764956
05 - May,1.500845,2.275755
06 - June,2.024375,1.866364
07 - July,1.867932,2.267310
08 - August,1.871564,1.639863
09 - September,2.095317,NaN


In [ ]:
# 1. Filtrar las columnas solicitadas del DataFrame comparativo de 2025-2026
columnas_deseadas = ['Fecha', 'Inflacion_Mensual (%)', 'Año', 'Mes']
df_filtrado_csv = df_comparativo[columnas_deseadas].copy()

# 2. Exportar a CSV
nombre_archivo_csv = "ipc_nacional_2025_2026.csv"
df_filtrado_csv.to_csv(nombre_archivo_csv, index=False, encoding='utf-8-sig')

# 3. Mostrar el resultado
print(f"Archivo '{nombre_archivo_csv}' generado con éxito con las columnas especificadas.")
display(df_filtrado_csv.head())

Archivo 'ipc_nacional_2025_2026.csv' generado con éxito con las columnas especificadas.


,Fecha,Inflacion_Mensual (%),Año,Mes
12,2025-01-01,2.023411,2025,01 - January
13,2025-02-01,2.234825,2025,02 - February
14,2025-03-01,3.854277,2025,03 - March
15,2025-04-01,2.757754,2025,04 - April
16,2025-05-01,1.500845,2025,05 - May


In [ ]:
import pandas as pd

# 1. Usar df_comparativo directamente de memoria (que contiene todas las columnas originales)
df_temp = df_comparativo.copy()

# 2. Crear la columna de periodo en texto (YYYY-MM)
df_temp['Periodo'] = pd.to_datetime(df_temp['Fecha']).dt.strftime('%Y-%m')

# 3. Pivotear dejando los periodos en las columnas horizontales
df_flourish = df_temp.pivot(index='Año', columns='Periodo', values='Inflacion_Mensual (%)').reset_index()

# 4. Renombrar la columna 'Año' a 'Label' para Flourish
df_flourish.rename(columns={'Año': 'Label'}, inplace=True)

# 5. Convertir a tipo texto la etiqueta 'Label'
df_flourish['Label'] = df_flourish['Label'].astype(str)

# 6. Guardar archivo directamente con el nombre solicitado: ipc_nacional_2025_2026.csv
df_flourish.to_csv("ipc_nacional_2025_2026.csv", index=False, encoding='utf-8-sig')

print("--- Estructura final horizontal guardada en 'ipc_nacional_2025_2026.csv' ---")
display(df_flourish)

--- Estructura final horizontal guardada en 'ipc_nacional_2025_2026.csv' ---


Periodo,Label,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,...,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08
0,2025,2.023411,2.234825,3.854277,2.757754,1.500845,2.024375,1.867932,1.871564,2.095317,...,2.481975,2.792697,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.768725,2.615914,3.410774,2.764956,2.275755,1.866364,2.26731,1.639863
